In [1]:
import sys
# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

In [2]:
import time
import json
import hashlib
import threading
import functions as fn
import dataTransfer as DT
import EncryptionDecryption as ED

In [3]:
with open('../Main/ReceivedData/keys.json', 'r') as file:
    keys = json.load(file)

In [4]:
with open('../Main/ReceivedData/presharedkeys.json', 'r') as file:
    presharedkeys = json.load(file)

presharedkeys = presharedkeys['key']

In [5]:
userNonce_i = fn.nonce_gen()

In [6]:
uniqueId_i = "RajeshDevice1"
uniqueId_g = "rajeshGateway"
uniqueId_j = "RajeshHomeDevice1"

In [7]:
temporalIdentity_j = hashlib.sha256((uniqueId_j + keys['device_public'] + str(userNonce_i)).encode()).hexdigest()
temporalIdentity_g = hashlib.sha256((uniqueId_g + keys['gateway_public'] + str(userNonce_i)).encode()).hexdigest()
temporalIdentity_i = hashlib.sha256((uniqueId_i + keys['user_public'] + str(userNonce_i)).encode()).hexdigest()

In [8]:
computeI = hashlib.sha256((temporalIdentity_i + temporalIdentity_j + temporalIdentity_g + str(userNonce_i)).encode()).hexdigest()

In [9]:
# step 1
fingerprintImagePath = "../Main/RequiredData/Fingerprint/11.jpg"
accelerometerDataPath = "../Main/RequiredData/Accelerometer/testingdataM55.csv"

In [10]:
imageHash = fn.hash_file(fingerprintImagePath)

accelerometerHash = fn.hash_file(accelerometerDataPath)

In [11]:
encryptedIdandImage = ED.symmetric_key_encryption('', 
                                                  ED.read_image_as_bytes(fingerprintImagePath), 
                                                  presharedkeys)

encryptedIDandAccelerometerData = ED.symmetric_key_encryption('',
                                                              ED.read_image_as_bytes(accelerometerDataPath), 
                                                              presharedkeys)

In [12]:
userDeviceData = {
    "encryptedImage" : encryptedIdandImage,
    "encryptedAccelerometerData" : encryptedIDandAccelerometerData,
    "imageHash" : imageHash,
    "accelerometerHash" : accelerometerHash,
    "userNonce" : userNonce_i,
    "compute" : computeI,
    "TDi" : temporalIdentity_i,
    "TDj" : temporalIdentity_j,
    "TDg" : temporalIdentity_g
}

In [13]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive,  args=("keyGeneration/authentication_user_send.json",))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(userDeviceData,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

Server is listening on port 12345...
Connected by ('127.0.0.1', 59826)
Data transfer done.
Received data saved to 'received_data.json'.
Data transfer and storage complete.


In [14]:
# step 2
print("Starting Step 2")
json_data = fn.read_json_file("../Main/ReceivedData/keyGeneration/authentication_user_send.json")

Starting Step 2


In [15]:
gatewayNonce = fn.nonce_gen()

In [16]:
decryptedIdandImage = ED.symmetric_key_decryption(json_data["encryptedImage"], 
                                                  presharedkeys)

decryptedIDandAccelerometerData = ED.symmetric_key_decryption(json_data["encryptedAccelerometerData"], 
                                                              presharedkeys)

print(f"both the above decryptions gave the same output unique id : {decryptedIdandImage[0] == decryptedIDandAccelerometerData[0]}")

print("image hash validated : ", hashlib.sha256(decryptedIdandImage[1]).hexdigest() == json_data["imageHash"])

both the above decryptions gave the same output unique id : True
image hash validated :  True


In [17]:
with open('../Main/ReceivedData/datastore.json', 'r') as file:
    datastore = json.load(file)

In [18]:
# saved by gatway
M_device = datastore[uniqueId_g]["smartDevices"][uniqueId_j]["secretIntegrity"]
M_user = datastore[uniqueId_g]["userAccessDevice"][uniqueId_i]["secretIntegrity"]

In [19]:
M = fn.xor_strings(M_user, M_device)
M = fn.xor_strings(M, str(gatewayNonce))

In [20]:
# both are saved by gateway
sharesGeneratedByGateway_i = 53
sharesGeneratedByGateway_j = 30

In [21]:
P_u = fn.xor_strings(str(gatewayNonce), str(sharesGeneratedByGateway_i))
P_u = fn.xor_strings(P_u, uniqueId_i)

P_d = fn.xor_strings(str(gatewayNonce), str(sharesGeneratedByGateway_j))
P_d = fn.xor_strings(P_d, uniqueId_j)

In [22]:
nonce_dg = fn.nonce_gen()
compute_j = hashlib.sha256((M + P_d + str(nonce_dg) + uniqueId_j).encode()).hexdigest()
compute_j = fn.xor_strings(compute_j, str(sharesGeneratedByGateway_j))


In [23]:
nonce_ug = fn.nonce_gen()
compute_i = hashlib.sha256((M + P_u + str(nonce_ug) + uniqueId_i).encode()).hexdigest()
compute_i = fn.xor_strings(compute_i, str(sharesGeneratedByGateway_i))

    #   validation in user side

In [24]:
data = {
    "M": M,
    "P_u": P_u,
    "nonce_ug": nonce_ug,
    "compute_i": compute_i
}
with open(f'./ReceivedData/keyGeneration/usershares.json', 'w') as json_file:
        json.dump(data, json_file, indent=4)

In [25]:
with open('./ReceivedData/keyGeneration/usershares.json', 'r') as file:
    usershares = json.load(file)

In [26]:
userhash = hashlib.sha256(
    (usershares["M"] +
    usershares["P_u"] +
    str(usershares["nonce_ug"]) +
    uniqueId_i).encode()
).hexdigest()

shares_u = fn.xor_strings(usershares["compute_i"], userhash)

In [31]:
userStoredSecret = datastore[uniqueId_g]["userAccessDevice"][uniqueId_i]["secretComputed"]

In [32]:
usershare = ((2 * int(shares_u)) - userStoredSecret) % fn.primeNumbergenerator()

In [35]:
hashlib.sha256((str(usershare) + uniqueId_g).encode()).hexdigest()

'a5f1f945f6b02339ca4707995ac5e998d500cf4245637cfdfccd7b4bbc4aef42'